In [3]:
import pandas as pd
import numpy as np
import os

for file in os.listdir("data/raw"):
    print(file)

.gitkeep
Cleaned_Viral_Social_Media_Trends.csv
Cleaned_Viral_Social_Media_Trends2.csv
INDIA_RETAIL_DATA.xlsx
Instagram_Analytics.csv
retail_sales_dataset.csv
youtube.csv


In [2]:
import os

# This shows exactly where Python is currently looking from
print(os.getcwd())

j:\Projects\consumer-behaviour-platform


In [5]:
# Loading all datasets into Pandas DataFrames
# pd.read_csv  → reads CSV files into a table Python can work with
# pd.read_excel → reads xlsx files (needs openpyxl library)
# We store each dataset in a variable ending with _df
# _df stands for DataFrame — the pandas table format

instagram_df = pd.read_csv("data/raw/Instagram_Analytics.csv")
trends_df    = pd.read_csv("data/raw/Cleaned_Viral_Social_Media_Trends.csv")
trends2_df   = pd.read_csv("data/raw/Cleaned_Viral_Social_Media_Trends2.csv")
retail_df    = pd.read_csv("data/raw/retail_sales_dataset.csv")
india_df     = pd.read_excel("data/raw/INDIA_RETAIL_DATA.xlsx")
youtube_df   = pd.read_csv("data/raw/youtube.csv")

print("All datasets loaded successfully")
print(f"Instagram rows    : {len(instagram_df)}")
print(f"Trends rows       : {len(trends_df)}")
print(f"Trends2 rows      : {len(trends2_df)}")
print(f"Retail rows       : {len(retail_df)}")
print(f"India retail rows : {len(india_df)}")
print(f"YouTube rows      : {len(youtube_df)}")

All datasets loaded successfully
Instagram rows    : 29999
Trends rows       : 5000
Trends2 rows      : 5000
Retail rows       : 1000
India retail rows : 2534
YouTube rows      : 161470


In [6]:
# .shape    → shows (rows, columns)
# .columns  → lists all column names
# .dtypes   → shows data type of each column
# .isnull().sum() → counts missing values per column
# .head(2)  → shows first 2 rows

# WHY WE DO THIS:
# Before cleaning anything we must understand what we have
# What columns exist, what is missing, what type each column is
# This is called data profiling — every analyst does this first

datasets = {
    "Instagram"    : instagram_df,
    "Trends"       : trends_df,
    "Trends2"      : trends2_df,
    "Retail"       : retail_df,
    "India Retail" : india_df,
    "YouTube"      : youtube_df
}

for name, df in datasets.items():
    print("=" * 60)
    print(f"DATASET: {name}")
    print(f"Shape  : {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nData types:\n{df.dtypes}")
    print(f"\nMissing values:\n{df.isnull().sum()}")
    print(f"\nFirst 2 rows:\n{df.head(2)}")
    print()

DATASET: Instagram
Shape  : (29999, 23)

Columns: ['post_id', 'account_id', 'account_type', 'follower_count', 'media_type', 'content_category', 'traffic_source', 'has_call_to_action', 'post_datetime', 'post_date', 'post_hour', 'day_of_week', 'likes', 'comments', 'shares', 'saves', 'reach', 'impressions', 'engagement_rate', 'followers_gained', 'caption_length', 'hashtags_count', 'performance_bucket_label']

Data types:
post_id                         str
account_id                    int64
account_type                    str
follower_count                int64
media_type                      str
content_category                str
traffic_source                  str
has_call_to_action            int64
post_datetime                   str
post_date                       str
post_hour                     int64
day_of_week                     str
likes                         int64
comments                      int64
shares                        int64
saves                         int64
re

In [8]:
# ============================================================
# PHASE 2 — DATA CLEANING
# ============================================================
# WHY WE CLEAN:
# Raw data always has issues — wrong types, inconsistent names,
# global data when we need local, missing categories.
# Cleaning makes data ready for analysis and AI features.
# Every step below has a specific reason explained in comments.
# ============================================================

# ── STEP 1: INSTAGRAM CLEANING ──────────────────────────────
# Convert date string to proper datetime so we can do
# time-based analysis like monthly trends
instagram_df["post_date"] = pd.to_datetime(instagram_df["post_date"])
instagram_df["post_datetime"] = pd.to_datetime(instagram_df["post_datetime"])

# Standardise text columns to lowercase for consistency
# Inconsistent casing causes groupby to treat "Brand" and "brand"
# as two different values — lowercase fixes this
instagram_df["account_type"]      = instagram_df["account_type"].str.lower().str.strip()
instagram_df["media_type"]        = instagram_df["media_type"].str.lower().str.strip()
instagram_df["content_category"]  = instagram_df["content_category"].str.lower().str.strip()
instagram_df["performance_bucket_label"] = instagram_df["performance_bucket_label"].str.lower().str.strip()

# Simulate age_group for each Instagram post
# Based on real platform demographics:
# Instagram is heavily used by 18-34 age group
# Brands target all groups but skew younger
np.random.seed(42)  # seed ensures reproducibility — same result every run
instagram_df["age_group"] = np.random.choice(
    ["13-17", "18-24", "25-34", "35-50", "50+"],
    size=len(instagram_df),
    p=[0.10, 0.40, 0.30, 0.15, 0.05]
)

# Simulate Chennai locality
# Weights based on real Chennai population and commercial density
chennai_localities = ["T Nagar", "Anna Nagar", "Velachery",
                      "Adyar", "OMR", "Tambaram", "Porur", "Nungambakkam"]
locality_weights   = [0.20, 0.15, 0.15, 0.12, 0.13, 0.10, 0.08, 0.07]

instagram_df["locality"] = np.random.choice(
    chennai_localities,
    size=len(instagram_df),
    p=locality_weights
)

print("✅ Instagram cleaned")
print(f"   Shape: {instagram_df.shape}")
print(f"   New columns added: age_group, locality")
print(f"   Sample age_group values: {instagram_df['age_group'].value_counts().to_dict()}")

✅ Instagram cleaned
   Shape: (29999, 25)
   New columns added: age_group, locality
   Sample age_group values: {'18-24': 11961, '25-34': 9054, '35-50': 4463, '13-17': 3028, '50+': 1493}


In [10]:
# ── STEP 2: TRENDS + TRENDS2 CLEANING ───────────────────────
# Both datasets have identical structure
# We combine them first to get 10,000 rows
# Then drop exact duplicate rows to keep only unique posts
# This gives us maximum data while removing redundancy

# Combine both trend datasets
trends_combined = pd.concat([trends_df, trends2_df], ignore_index=True)
print(f"Combined rows before dedup : {len(trends_combined)}")

# Drop exact duplicate rows
# A duplicate means every single column value is identical
# Keeping duplicates would skew our analysis — same post counted twice
trends_combined = trends_combined.drop_duplicates()
print(f"Combined rows after dedup  : {len(trends_combined)}")

# Convert date to datetime
trends_combined["Post_Date"] = pd.to_datetime(trends_combined["Post_Date"])

# Standardise column names to snake_case
# snake_case means all lowercase with underscores
# This makes coding easier — no spaces, consistent style
trends_combined.columns = [
    "post_id", "post_date", "platform", "hashtag",
    "content_type", "region", "views", "likes",
    "shares", "comments", "engagement_level"
]

# Clean hashtag column — remove the # symbol
# We keep just the word: #Challenge → Challenge
# Cleaner for groupby and visualisation
trends_combined["hashtag"] = trends_combined["hashtag"].str.replace("#", "", regex=False).str.strip()

# Standardise text to lowercase
trends_combined["platform"]         = trends_combined["platform"].str.lower().str.strip()
trends_combined["content_type"]     = trends_combined["content_type"].str.lower().str.strip()
trends_combined["region"]           = trends_combined["region"].str.lower().str.strip()
trends_combined["engagement_level"] = trends_combined["engagement_level"].str.lower().str.strip()
trends_combined["hashtag"]          = trends_combined["hashtag"].str.lower().str.strip()

# Filter for India only
# Our project is Chennai focused — global data adds noise
# We keep India rows for primary analysis
trends_india = trends_combined[trends_combined["region"] == "india"].copy()
print(f"India only rows            : {len(trends_india)}")

# If India rows are too few we keep all regions but flag it
# This is a safety check — we never want to lose too much data
if len(trends_india) < 200:
    print("⚠️  India rows too few — keeping all regions for analysis")
    trends_india = trends_combined.copy()
else:
    print("✅ India filter applied successfully")

# Simulate age_group for trends
# TikTok and Instagram lean younger, YouTube more spread out
def simulate_age_by_platform(platform):
    if platform in ["tiktok", "instagram"]:
        return np.random.choice(
            ["13-17", "18-24", "25-34", "35-50", "50+"],
            p=[0.15, 0.40, 0.28, 0.12, 0.05]
        )
    elif platform == "youtube":
        return np.random.choice(
            ["13-17", "18-24", "25-34", "35-50", "50+"],
            p=[0.10, 0.30, 0.30, 0.20, 0.10]
        )
    else:  # twitter, facebook
        return np.random.choice(
            ["13-17", "18-24", "25-34", "35-50", "50+"],
            p=[0.05, 0.25, 0.35, 0.25, 0.10]
        )

np.random.seed(42)
trends_combined["age_group"] = trends_combined["platform"].apply(simulate_age_by_platform)
trends_india["age_group"]    = trends_india["platform"].apply(simulate_age_by_platform)

# Simulate Chennai locality
trends_combined["locality"] = np.random.choice(
    chennai_localities, size=len(trends_combined), p=locality_weights
)
trends_india["locality"] = np.random.choice(
    chennai_localities, size=len(trends_india), p=locality_weights
)

print("\n✅ Trends cleaned")
print(f"   Combined shape : {trends_combined.shape}")
print(f"   India shape    : {trends_india.shape}")
print(f"   Platforms      : {trends_combined['platform'].unique()}")
print(f"   Content types  : {trends_combined['content_type'].unique()}")

Combined rows before dedup : 10000
Combined rows after dedup  : 5000
India only rows            : 617
✅ India filter applied successfully

✅ Trends cleaned
   Combined shape : (5000, 13)
   India shape    : (617, 13)
   Platforms      : <ArrowStringArray>
['tiktok', 'instagram', 'twitter', 'youtube']
Length: 4, dtype: str
   Content types  : <ArrowStringArray>
['video', 'shorts', 'post', 'tweet', 'live stream', 'reel']
Length: 6, dtype: str


In [11]:
# ── STEP 3: RETAIL SALES CLEANING ───────────────────────────
# This is our core BUYING WORLD dataset
# It has Age which is our bridge column to social media data
# Main tasks: rename columns, derive age_group,
# simulate platform and locality, convert date

# Rename columns — remove spaces, use snake_case
# Spaces in column names cause errors in many operations
# snake_case is the professional Python standard
retail_df.columns = [
    "transaction_id", "date", "customer_id", "gender",
    "age", "product_category", "quantity",
    "price_per_unit", "total_amount"
]

# Convert date to datetime
retail_df["date"] = pd.to_datetime(retail_df["date"])

# Standardise text columns
retail_df["gender"]           = retail_df["gender"].str.lower().str.strip()
retail_df["product_category"] = retail_df["product_category"].str.lower().str.strip()

# Derive age_group from age
# This is feature engineering — creating a new useful column
# from an existing one. Age groups let us merge with
# social media data which also has age_group
def age_to_group(age):
    if age <= 17:   return "13-17"
    elif age <= 24: return "18-24"
    elif age <= 34: return "25-34"
    elif age <= 50: return "35-50"
    else:           return "50+"

retail_df["age_group"] = retail_df["age"].apply(age_to_group)

# Simulate platform based on age group
# This is the KEY bridge column — which platform does
# this buyer use? Based on real world usage patterns
def simulate_platform(age_group):
    platforms = {
        "13-17": (["instagram", "tiktok", "youtube"],         [0.40, 0.40, 0.20]),
        "18-24": (["instagram", "tiktok", "youtube"],         [0.40, 0.35, 0.25]),
        "25-34": (["instagram", "youtube", "twitter"],        [0.35, 0.35, 0.30]),
        "35-50": (["youtube",   "twitter", "instagram"],      [0.40, 0.35, 0.25]),
        "50+":   (["youtube",   "twitter", "instagram"],      [0.50, 0.30, 0.20]),
    }
    choices, weights = platforms[age_group]
    return np.random.choice(choices, p=weights)

np.random.seed(42)
retail_df["platform"] = retail_df["age_group"].apply(simulate_platform)

# Simulate top content genre watched based on product category
# This is the CORE INSIGHT column — what content leads to what purchase
# Based on known marketing and consumer behaviour patterns:
# Fashion buyers watch style/beauty content
# Electronics buyers watch tech content
# Food buyers watch food/lifestyle content
def simulate_content_genre(product_category):
    genre_map = {
        "electronics": (["video", "shorts", "reel"],          [0.50, 0.30, 0.20]),
        "clothing":    (["reel",  "shorts", "video"],         [0.45, 0.35, 0.20]),
        "beauty":      (["reel",  "shorts", "live stream"],   [0.45, 0.30, 0.25]),
        "food":        (["shorts","video",  "live stream"],   [0.40, 0.35, 0.25]),
        "sports":      (["video", "shorts", "reel"],          [0.50, 0.30, 0.20]),
        "books":       (["video", "post",   "tweet"],         [0.50, 0.30, 0.20]),
    }
    # Default if category not in map
    if product_category not in genre_map:
        return np.random.choice(["video", "shorts", "reel"], p=[0.40, 0.35, 0.25])
    choices, weights = genre_map[product_category]
    return np.random.choice(choices, p=weights)

retail_df["top_content_genre"] = retail_df["product_category"].apply(simulate_content_genre)

# Simulate daily screen time based on age group
# Younger users spend more time on social media
# Based on real average screen time studies
def simulate_screen_time(age_group):
    screen_time_ranges = {
        "13-17": (3.0, 6.0),
        "18-24": (3.0, 5.5),
        "25-34": (2.0, 4.5),
        "35-50": (1.5, 3.5),
        "50+":   (1.0, 2.5),
    }
    low, high = screen_time_ranges[age_group]
    return round(np.random.uniform(low, high), 1)

retail_df["daily_screen_time_hrs"] = retail_df["age_group"].apply(simulate_screen_time)

# Simulate Chennai locality
retail_df["locality"] = np.random.choice(
    chennai_localities, size=len(retail_df), p=locality_weights
)

# Simulate impulse buy frequency based on age and screen time
# Higher screen time + younger age = more impulse buying
# Based on consumer psychology research
def simulate_impulse_buy(row):
    score = 0
    if row["age_group"] in ["13-17", "18-24"]: score += 2
    elif row["age_group"] == "25-34":          score += 1
    if row["daily_screen_time_hrs"] >= 4.0:    score += 2
    elif row["daily_screen_time_hrs"] >= 2.5:  score += 1
    if score >= 3:   return "often"
    elif score >= 1: return "sometimes"
    else:            return "rarely"

retail_df["impulse_buy_freq"] = retail_df.apply(simulate_impulse_buy, axis=1)

print("✅ Retail cleaned")
print(f"   Shape: {retail_df.shape}")
print(f"   Age groups      : {retail_df['age_group'].value_counts().to_dict()}")
print(f"   Platforms       : {retail_df['platform'].value_counts().to_dict()}")
print(f"   Categories      : {retail_df['product_category'].unique()}")
print(f"   Impulse buy     : {retail_df['impulse_buy_freq'].value_counts().to_dict()}")

✅ Retail cleaned
   Shape: (1000, 15)
   Age groups      : {'35-50': 335, '50+': 313, '25-34': 203, '18-24': 149}
   Platforms       : {'youtube': 415, 'instagram': 282, 'twitter': 255, 'tiktok': 48}
   Categories      : <ArrowStringArray>
['beauty', 'clothing', 'electronics']
Length: 3, dtype: str
   Impulse buy     : {'rarely': 462, 'sometimes': 352, 'often': 186}


In [7]:
# Running inspection one by one to avoid truncation
# We check each dataset separately so nothing gets cut off

print("=" * 60)
print("1. INSTAGRAM")
print("Shape:", instagram_df.shape)
print("\nColumns:", instagram_df.columns.tolist())
print("\nMissing values:\n", instagram_df.isnull().sum())
print("\nSample row:\n", instagram_df.head(1).to_dict())

1. INSTAGRAM
Shape: (29999, 23)

Columns: ['post_id', 'account_id', 'account_type', 'follower_count', 'media_type', 'content_category', 'traffic_source', 'has_call_to_action', 'post_datetime', 'post_date', 'post_hour', 'day_of_week', 'likes', 'comments', 'shares', 'saves', 'reach', 'impressions', 'engagement_rate', 'followers_gained', 'caption_length', 'hashtags_count', 'performance_bucket_label']

Missing values:
 post_id                     0
account_id                  0
account_type                0
follower_count              0
media_type                  0
content_category            0
traffic_source              0
has_call_to_action          0
post_datetime               0
post_date                   0
post_hour                   0
day_of_week                 0
likes                       0
comments                    0
shares                      0
saves                       0
reach                       0
impressions                 0
engagement_rate             0
followers_g

In [12]:
# ── STEP 4: INDIA RETAIL CLEANING ───────────────────────────
# This dataset gives us Indian city and state level data
# We use it purely for locality and regional patterns
# Main task: extract Tamil Nadu rows, clean column names,
# map cities to Chennai localities where possible

# Rename columns to snake_case
india_df.columns = [
    "order_priority", "discount", "unit_price", "freight_expenses",
    "freight_mode", "segment", "product_type", "product_sub_category",
    "product_container", "state", "city", "region",
    "country", "order_date", "ship_date", "profit",
    "qty_ordered", "sales"
]

# Standardise text columns
india_df["state"]        = india_df["state"].str.lower().str.strip()
india_df["city"]         = india_df["city"].str.lower().str.strip()
india_df["region"]       = india_df["region"].str.lower().str.strip()
india_df["segment"]      = india_df["segment"].str.lower().str.strip()
india_df["product_type"] = india_df["product_type"].str.lower().str.strip()
india_df["freight_mode"] = india_df["freight_mode"].str.lower().str.strip()

# Filter for Tamil Nadu / South India
# Our project is Chennai focused
# South region gives us the most relevant data
india_tn = india_df[
    (india_df["state"] == "tamil nadu") |
    (india_df["region"] == "south")
].copy()

print(f"Tamil Nadu rows  : {len(india_df[india_df['state'] == 'tamil nadu'])}")
print(f"South region rows: {len(india_df[india_df['region'] == 'south'])}")
print(f"Combined TN+South: {len(india_tn)}")

# If TN rows are too few keep all India data
if len(india_tn) < 100:
    print("⚠️  Too few TN rows — keeping all India data")
    india_tn = india_df.copy()
else:
    print("✅ Tamil Nadu / South filter applied")

# Add locality column based on city
# Map known Tamil Nadu cities to Chennai localities
# Cities not in Tamil Nadu get a simulated Chennai locality
city_to_locality = {
    "chennai":    "T Nagar",
    "coimbatore": "Anna Nagar",
    "madurai":    "Velachery",
    "salem":      "Tambaram",
    "trichy":     "Porur",
    "tirunelveli":"OMR",
}

def map_locality(city):
    if city in city_to_locality:
        return city_to_locality[city]
    return np.random.choice(chennai_localities, p=locality_weights)

india_tn["locality"] = india_tn["city"].apply(map_locality)

# Add simulated age_group — India retail data has no age column
# Use segment to infer — corporate = older, consumer = mixed
def segment_to_age(segment):
    if "corporate" in segment or "hotel" in segment:
        return np.random.choice(
            ["25-34", "35-50", "50+"], p=[0.30, 0.45, 0.25]
        )
    else:
        return np.random.choice(
            ["18-24", "25-34", "35-50"], p=[0.35, 0.40, 0.25]
        )

np.random.seed(42)
india_tn["age_group"] = india_tn["segment"].apply(segment_to_age)

print("\n✅ India Retail cleaned")
print(f"   Shape         : {india_tn.shape}")
print(f"   States present: {india_tn['state'].unique()[:8]}")
print(f"   Cities present: {india_tn['city'].unique()[:8]}")
print(f"   Age groups    : {india_tn['age_group'].value_counts().to_dict()}")

Tamil Nadu rows  : 184
South region rows: 542
Combined TN+South: 542
✅ Tamil Nadu / South filter applied

✅ India Retail cleaned
   Shape         : (542, 20)
   States present: <ArrowStringArray>
['kerala', 'andhra pradesh', 'tamil nadu', 'telangana', 'karnataka']
Length: 5, dtype: str
   Cities present: <ArrowStringArray>
['pathanamthitta',   'vizianagaram',     'coimbatore',     'trivandrum',
        'vellore',     'perambalur',      'kozhikode',       'pollachi']
Length: 8, dtype: str
   Age groups    : {'35-50': 195, '25-34': 189, '18-24': 106, '50+': 52}


In [13]:
# ── STEP 5: YOUTUBE CLEANING ────────────────────────────────
# YouTube has 161,470 rows — too large for our project
# We sample 10,000 rows to keep it manageable
# Main tasks: sample, map category_id to names,
# clean date, simulate age_group and locality

# Sample 10,000 rows randomly
# random_state=42 ensures same sample every run
youtube_sample = youtube_df.sample(n=10000, random_state=42).copy()
youtube_sample = youtube_sample.reset_index(drop=True)
print(f"YouTube sampled : {len(youtube_sample)} rows")

# Map category_id numbers to readable category names
# category_id is just a number — meaningless for analysis
# We need the actual genre name for our correlation work
youtube_categories = {
    1:  "Film & Animation",
    2:  "Autos & Vehicles",
    10: "Music",
    15: "Pets & Animals",
    17: "Sports",
    19: "Travel & Events",
    20: "Gaming",
    22: "People & Blogs",
    23: "Comedy",
    24: "Entertainment",
    25: "News & Politics",
    26: "Howto & Style",
    27: "Education",
    28: "Science & Technology",
    29: "Nonprofits & Activism"
}

youtube_sample["content_category"] = youtube_sample["category_id"].map(youtube_categories)

# Check if any category_ids were not mapped
unmapped = youtube_sample["content_category"].isnull().sum()
print(f"Unmapped categories: {unmapped}")

# Fill any unmapped with Entertainment as default
youtube_sample["content_category"] = youtube_sample["content_category"].fillna("Entertainment")

# Standardise text columns
youtube_sample["content_category"]      = youtube_sample["content_category"].str.lower().str.strip()
youtube_sample["published_day_of_week"] = youtube_sample["published_day_of_week"].str.lower().str.strip()
youtube_sample["publish_country"]       = youtube_sample["publish_country"].str.lower().str.strip()

# Filter for US and India — most relevant regions
# US gives us strong trend data, India gives local relevance
youtube_filtered = youtube_sample[
    youtube_sample["publish_country"].isin(["us", "in"])
].copy()

print(f"US + India rows : {len(youtube_filtered)}")

# If too few keep all
if len(youtube_filtered) < 500:
    print("⚠️  Too few US/India rows — keeping all countries")
    youtube_filtered = youtube_sample.copy()
else:
    print("✅ US + India filter applied")

# Simulate age_group based on content category
# Different YouTube content attracts different age groups
# Based on YouTube demographic research
def simulate_yt_age(category):
    if category in ["gaming", "music", "entertainment", "comedy"]:
        return np.random.choice(
            ["13-17", "18-24", "25-34", "35-50", "50+"],
            p=[0.20, 0.40, 0.25, 0.10, 0.05]
        )
    elif category in ["education", "science & technology", "howto & style"]:
        return np.random.choice(
            ["13-17", "18-24", "25-34", "35-50", "50+"],
            p=[0.10, 0.30, 0.35, 0.20, 0.05]
        )
    elif category in ["news & politics", "people & blogs"]:
        return np.random.choice(
            ["13-17", "18-24", "25-34", "35-50", "50+"],
            p=[0.05, 0.20, 0.30, 0.30, 0.15]
        )
    else:
        return np.random.choice(
            ["13-17", "18-24", "25-34", "35-50", "50+"],
            p=[0.10, 0.30, 0.30, 0.20, 0.10]
        )

np.random.seed(42)
youtube_filtered["age_group"] = youtube_filtered["content_category"].apply(simulate_yt_age)

# Simulate locality
youtube_filtered["locality"] = np.random.choice(
    chennai_localities,
    size=len(youtube_filtered),
    p=locality_weights
)

# Add platform column for consistency across all datasets
youtube_filtered["platform"] = "youtube"

# Select only columns we need for analysis
youtube_clean = youtube_filtered[[
    "video_id", "title", "channel_title", "content_category",
    "publish_country", "published_day_of_week",
    "views", "likes", "dislikes", "comment_count",
    "age_group", "locality", "platform"
]].copy()

print("\n✅ YouTube cleaned")
print(f"   Shape              : {youtube_clean.shape}")
print(f"   Top categories     : {youtube_clean['content_category'].value_counts().head(5).to_dict()}")
print(f"   Age groups         : {youtube_clean['age_group'].value_counts().to_dict()}")
print(f"   Countries          : {youtube_clean['publish_country'].value_counts().to_dict()}")

YouTube sampled : 10000 rows
Unmapped categories: 18
US + India rows : 2551
✅ US + India filter applied

✅ YouTube cleaned
   Shape              : (2551, 13)
   Top categories     : {'entertainment': 634, 'music': 403, 'howto & style': 248, 'people & blogs': 223, 'comedy': 220}
   Age groups         : {'18-24': 828, '25-34': 735, '35-50': 404, '13-17': 391, '50+': 193}
   Countries          : {'us': 2551}


In [14]:
# ── STEP 6: SAVE ALL CLEANED DATASETS ───────────────────────
# We save every cleaned dataset to data/cleaned folder
# WHY: Raw data should never be modified directly
# The cleaned folder holds our processed versions
# If anything goes wrong we always have the raw originals
# This is standard professional data practice

instagram_df.to_csv("data/cleaned/instagram_clean.csv", index=False)
trends_combined.to_csv("data/cleaned/trends_clean.csv", index=False)
trends_india.to_csv("data/cleaned/trends_india_clean.csv", index=False)
retail_df.to_csv("data/cleaned/retail_clean.csv", index=False)
india_tn.to_csv("data/cleaned/india_retail_clean.csv", index=False)
youtube_clean.to_csv("data/cleaned/youtube_clean.csv", index=False)

print("✅ All cleaned datasets saved to data/cleaned/")
print()

# Verify files were created
import os
for f in os.listdir("data/cleaned"):
    size = os.path.getsize(f"data/cleaned/{f}")
    print(f"   {f:40s} {size/1024:.1f} KB")

✅ All cleaned datasets saved to data/cleaned/

   .gitkeep                                 0.0 KB
   india_retail_clean.csv                   95.3 KB
   instagram_clean.csv                      4478.2 KB
   retail_clean.csv                         90.1 KB
   trends_clean.csv                         469.7 KB
   trends_india_clean.csv                   57.8 KB
   youtube_clean.csv                        362.6 KB


In [15]:
# ── STEP 7: BUILD MASTER MERGED TABLE ───────────────────────
# This is the most important step in the entire project
# We merge the social media world and buying world
# using age_group as the bridge column
# WHY: This single table powers every chart, every AI
# prediction, and every insight in our Streamlit app

# Step 7a — Summarise social media by age group
# We find the most common platform and content type
# per age group across all social media datasets
social_summary = trends_combined.groupby("age_group").agg(
    top_platform    = ("platform",         lambda x: x.value_counts().index[0]),
    top_content     = ("content_type",     lambda x: x.value_counts().index[0]),
    avg_views       = ("views",            "mean"),
    avg_likes       = ("likes",            "mean"),
    avg_engagement  = ("engagement_level", lambda x: x.value_counts().index[0]),
    total_posts     = ("post_id",          "count")
).reset_index()

print("Social summary by age group:")
print(social_summary)
print()

# Step 7b — Summarise retail by age group
retail_summary = retail_df.groupby("age_group").agg(
    top_purchase_category = ("product_category",    lambda x: x.value_counts().index[0]),
    avg_spend             = ("total_amount",         "mean"),
    avg_screen_time       = ("daily_screen_time_hrs","mean"),
    top_impulse_freq      = ("impulse_buy_freq",     lambda x: x.value_counts().index[0]),
    total_transactions    = ("transaction_id",       "count")
).reset_index()

print("Retail summary by age group:")
print(retail_summary)
print()

# Step 7c — Merge both summaries on age_group
# This creates the core insight table
master_df = social_summary.merge(retail_summary, on="age_group", how="inner")

# Step 7d — Add locality level detail from instagram
# Instagram has both age_group and locality
# This lets us add a location dimension to our master table
locality_summary = instagram_df.groupby(
    ["age_group", "locality"]
).agg(
    avg_reach        = ("reach",           "mean"),
    avg_impressions  = ("impressions",     "mean"),
    avg_eng_rate     = ("engagement_rate", "mean"),
    post_count       = ("post_id",         "count")
).reset_index()

# Save master tables
master_df.to_csv("data/cleaned/master_merged.csv", index=False)
locality_summary.to_csv("data/cleaned/locality_summary.csv", index=False)

print("✅ Master merged table built")
print(f"   Shape: {master_df.shape}")
print()
print("Master table preview:")
print(master_df.to_string())
print()
print("✅ Locality summary built")
print(f"   Shape: {locality_summary.shape}")
print()

# Final summary of everything we have built
print("=" * 55)
print("PHASE 2 COMPLETE — ALL DATASETS READY")
print("=" * 55)
print(f"   instagram_clean    : {len(instagram_df):>6} rows, {instagram_df.shape[1]} cols")
print(f"   trends_clean       : {len(trends_combined):>6} rows, {trends_combined.shape[1]} cols")
print(f"   trends_india_clean : {len(trends_india):>6} rows, {trends_india.shape[1]} cols")
print(f"   retail_clean       : {len(retail_df):>6} rows, {retail_df.shape[1]} cols")
print(f"   india_retail_clean : {len(india_tn):>6} rows, {india_tn.shape[1]} cols")
print(f"   youtube_clean      : {len(youtube_clean):>6} rows, {youtube_clean.shape[1]} cols")
print(f"   master_merged      : {len(master_df):>6} rows, {master_df.shape[1]} cols")
print(f"   locality_summary   : {len(locality_summary):>6} rows, {locality_summary.shape[1]} cols")
print()
print("Next → Phase 3: EDA + Sentiment Analysis")

Social summary by age group:
  age_group top_platform  top_content     avg_views      avg_likes  \
0     13-17    instagram         reel  2.440168e+06  245660.132867   
1     18-24       tiktok        video  2.506604e+06  255696.613809   
2     25-34      twitter         post  2.456688e+06  249524.183523   
3     35-50      twitter  live stream  2.563239e+06  251113.788927   
4       50+      youtube        tweet  2.508652e+06  249628.910864   

  avg_engagement  total_posts  
0           high          572  
1            low         1709  
2            low         1493  
3         medium          867  
4           high          359  

Retail summary by age group:
  age_group top_purchase_category   avg_spend  avg_screen_time  \
0     18-24                beauty  501.006711         4.305369   
1     25-34              clothing  478.275862         3.159113   
2     35-50              clothing  450.597015         2.522687   
3       50+              clothing  425.910543         1.764217  